# Exp5 Part B — Train + inference + submission

**Needs:** Part A outputs (`test_*_name/addr/union.tsv`) — on local disk if this is
the same runtime, else auto-restored from Drive in cell 1.

**You run:** cells top to bottom, then upload to the Portal.
Time: subset blocking ~1–2 h, train ~1 h, inference ~2–3 h (unattended).

In [ ]:
# 0) Setup: Drive, repo (clone once, pull after), data, deps. (~15-25 min first time)
BRANCH = "arena/01a0e27b-ml-challenge"
from google.colab import drive
drive.mount('/content/drive')
!apt-get -qq update && apt-get -qq install -y git-lfs > /dev/null 2>&1
!git lfs install > /dev/null 2>&1
!if [ -d /content/ML-challenge ]; then cd /content/ML-challenge && git checkout {BRANCH} && git pull --ff-only; else git clone -b {BRANCH} https://github.com/Saadmadni84/ML-challenge.git /content/ML-challenge; fi
%cd /content/ML-challenge
!git lfs pull
!wc -l dataset/train/*.tsv dataset/test/*.tsv
!pip -q install -r code/business_entity_resolution/requirements.txt
!python3 -c "import sklearn, lightgbm, rapidfuzz, sparse_dot_topn; print('deps OK')"
# Countries are DERIVED from the data (open set - never hardcoded).
import csv, os
from collections import Counter
os.makedirs("output", exist_ok=True)
c = Counter()
with open("dataset/test/test_source1.tsv", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f, delimiter="\t"):
        c[(row.get("country") or "").strip()] += 1
COUNTRIES = sorted(c)
print("test S1 countries:", dict(c))
assert all(COUNTRIES), "blank country label found in test_source1.tsv!"


In [ ]:
# 1) Restore Part-A outputs from Drive when missing locally.
import os, shutil
need = ([f"output/test_{C}_{F}.tsv" for C in COUNTRIES for F in ["name", "address"]]
        + [f"output/test_union_{C}.tsv" for C in COUNTRIES])
missing = [p for p in need if not (os.path.isfile(p) and os.path.getsize(p) > 100)]
print("missing:", missing if missing else "NONE - all local")
for p in missing:
    src = "/content/drive/MyDrive/" + os.path.basename(p)
    assert os.path.isfile(src), f"NOT ON DRIVE EITHER: {src} -> finish Part A first!"
    shutil.copy(src, p)
!ls -lh output/test_*.tsv


In [ ]:
# 2) Train/val SUBSET ids (stratified 200k/20k) + S1/GT slice files -> subset/.
import os
import pandas as pd
os.makedirs("subset", exist_ok=True)
s1 = pd.read_csv("dataset/train/train_source1.tsv", sep="\t")
gt = pd.read_csv("dataset/train/train_ground_truth.tsv", sep="\t")
gt["k"] = gt["matched_entity_ids"].fillna("").apply(lambda s: 0 if s == "" else s.count(",") + 1)
df = gt.merge(s1[["entity_id", "country"]], left_on="source1_entity_id", right_on="entity_id")
df["stratum"] = df["country"] + "_" + df["k"].clip(upper=6).astype(str)
tr = df.groupby("stratum", group_keys=False).apply(lambda g: g.sample(frac=200000 / len(df), random_state=42))
rest = df.drop(tr.index)
va = rest.groupby("stratum", group_keys=False).apply(lambda g: g.sample(frac=20000 / len(rest), random_state=43))
print("train", len(tr), "val", len(va))
for name, sub in [("train200k", tr), ("val20k", va)]:
    sub[["source1_entity_id", "matched_entity_ids"]].to_csv(f"subset/{name}_gt.tsv", sep="\t", index=False)
    s1[s1["entity_id"].isin(set(sub["source1_entity_id"]))].to_csv(f"subset/{name}_s1.tsv", sep="\t", index=False)
sub = pd.concat([s1[s1["entity_id"].isin(set(tr["source1_entity_id"]))],
                 s1[s1["entity_id"].isin(set(va["source1_entity_id"]))]])
sub.to_csv("subset/sub220k_s1.tsv", sep="\t", index=False)
print(sub["country"].value_counts().to_dict())


In [ ]:
# 3) Train-subset blocking vs FULL train targets (220k queries x 2 fields), then split by id.
import glob, subprocess
import pandas as pd
S = "code/business_entity_resolution/src"
for F in ["name", "address"]:
    subprocess.run(["python3", f"{S}/blocking.py",
                    "--s1", "subset/sub220k_s1.tsv",
                    "--s2", "dataset/train/train_source2.tsv",
                    "--s3", "dataset/train/train_source3.tsv",
                    "--field", F, "--out", f"output/sub220k_{F}_{{country}}.tsv",
                    "--k", "40", "--min-ngram", "3", "--max-ngram", "4",
                    "--min-df", "2", "--max-df", "0.05",
                    "--countries", "all", "--resume"], check=True)
SUB_C = sorted(glob.glob("output/sub220k_name_*.tsv"))
SUB_C = [p.rsplit("_", 1)[1][:-4] for p in SUB_C]
print("subset countries:", SUB_C)
for C in SUB_C:
    subprocess.run(["python3", f"{S}/union_candidates.py",
                    "--a", f"output/sub220k_name_{C}.tsv",
                    "--b", f"output/sub220k_addr_{C}.tsv",
                    "--k-per-file", "40",
                    "--out", f"output/sub220k_union_{C}.tsv"], check=True)
# split union+channels into train200k / val20k id sets (single combined files)
tr_ids = set(pd.read_csv("subset/train200k_gt.tsv", sep="\t")["source1_entity_id"])
va_ids = set(pd.read_csv("subset/val20k_gt.tsv", sep="\t")["source1_entity_id"])
for kind in ["union", "name", "addr"]:
    df = pd.concat([pd.read_csv(f"output/sub220k_{kind}_{C}.tsv", sep="\t") for C in SUB_C], ignore_index=True)
    df[df["source1_entity_id"].isin(tr_ids)].to_csv(f"output/cands_train200k_{kind}.tsv", sep="\t", index=False)
    df[df["source1_entity_id"].isin(va_ids)].to_csv(f"output/cands_val20k_{kind}.tsv", sep="\t", index=False)
    print(kind, len(df))


In [ ]:
# 4) SCALE-RECALL CHECK: full-target recall on val20k. PASTE OUTPUT. Expect ~0.99.
import subprocess
S = "code/business_entity_resolution/src"
subprocess.run(["python3", f"{S}/blocking_recall.py",
                "--candidates", "output/cands_val20k_union.tsv",
                "--ground-truth", "subset/val20k_gt.tsv",
                "--s1", "subset/val20k_s1.tsv",
                "--s2", "dataset/train/train_source2.tsv",
                "--s3", "dataset/train/train_source3.tsv",
                "--ks", "25,50,80"], check=True)


In [ ]:
# 5) TRAIN on 200k (~1h) -> model + tuned tau. PASTE threshold.json + BEST line.
import shutil, subprocess
S = "code/business_entity_resolution/src"
subprocess.run(["python3", f"{S}/train.py",
                "--s1", "subset/sub220k_s1.tsv",
                "--s2", "dataset/train/train_source2.tsv",
                "--s3", "dataset/train/train_source3.tsv",
                "--train-gt", "subset/train200k_gt.tsv",
                "--val-gt", "subset/val20k_gt.tsv",
                "--candidates", "output/cands_train200k_union.tsv",
                "--cand-name", "output/cands_train200k_name.tsv",
                "--cand-addr", "output/cands_train200k_addr.tsv",
                "--feature-set", "v2", "--out-dir", "output/full_model",
                "--seed", "42"], check=True)
!cat output/full_model/threshold.json
shutil.copy("output/full_model/model.txt", "/content/drive/MyDrive/model.txt")
shutil.copy("output/full_model/threshold.json", "/content/drive/MyDrive/threshold.json")


In [ ]:
# 6) TEST INFERENCE per country (~2-3h) -> concat -> VALIDATE. Leave tab open.
# predict.py streams S1 + candidates in lockstep, so each per-country union
# needs its matching per-country S1 slice (same row order as test_source1.tsv).
import csv, shutil, subprocess
from collections import Counter
S = "code/business_entity_resolution/src"

# 6a) Per-country S1 slices (streaming, order-preserving).
counts = Counter()
writers, files = {}, {}
with open("dataset/test/test_source1.tsv", encoding="utf-8", newline="") as f:
    rdr = csv.DictReader(f, delimiter="\t")
    for C in COUNTRIES:
        fh = open(f"output/test_s1_{C}.tsv", "w", encoding="utf-8", newline="")
        w = csv.DictWriter(fh, fieldnames=rdr.fieldnames, delimiter="\t")
        w.writeheader()
        files[C], writers[C] = fh, w
    for row in rdr:
        cc = (row.get("country") or "").strip()
        assert cc in writers, f"unexpected test country {cc!r} - add it to COUNTRIES and re-run Part A!"
        writers[cc].writerow(row)
        counts[cc] += 1
for fh in files.values():
    fh.close()
print("sliced test S1 rows:", dict(counts))

# 6b) Predict per country + Drive backup.
for C in COUNTRIES:
    subprocess.run(["python3", f"{S}/predict.py",
                    "--s1", f"output/test_s1_{C}.tsv",
                    "--s2", "dataset/test/test_source2.tsv",
                    "--s3", "dataset/test/test_source3.tsv",
                    "--candidates", f"output/test_union_{C}.tsv",
                    "--cand-name", f"output/test_{C}_name.tsv",
                    "--cand-addr", f"output/test_{C}_addr.tsv",
                    "--model", "output/full_model/model.txt",
                    "--threshold-json", "output/full_model/threshold.json",
                    "--feature-set", "v2", "--country", C, "--workers", "2",
                    "--out-matches", f"output/matching_{C}.tsv",
                    "--out-cands", f"output/candpairs_{C}.tsv"], check=True)
    shutil.copy(f"output/matching_{C}.tsv", "/content/drive/MyDrive/")
    shutil.copy(f"output/candpairs_{C}.tsv", "/content/drive/MyDrive/")

# 6c) Concat shards -> submission files (order across shards does not matter).
def concat_parts(parts, out):
    with open(out, "w", encoding="utf-8", newline="") as fo:
        for i, p in enumerate(parts):
            with open(p, encoding="utf-8", newline="") as fi:
                h = fi.readline()
                assert h.split("\t")[0] == "source1_entity_id", p
                if i == 0:
                    fo.write(h)
                shutil.copyfileobj(fi, fo)
concat_parts([f"output/matching_{C}.tsv" for C in COUNTRIES], "output/matching_results.tsv")
concat_parts([f"output/candpairs_{C}.tsv" for C in COUNTRIES], "output/candidate_pairs.tsv")
!wc -l output/matching_results.tsv output/candidate_pairs.tsv
subprocess.run(["python3", "utils/validate_submission.py",
                "--matching", "output/matching_results.tsv",
                "--candidate", "output/candidate_pairs.tsv",
                "--test-dir", "dataset/test"], check=True)
shutil.copy("output/matching_results.tsv", "/content/drive/MyDrive/")
shutil.copy("output/candidate_pairs.tsv", "/content/drive/MyDrive/")
print("VALIDATED + backed up - ready for Portal upload.")


In [ ]:
# 7) Sanity stats + download. PASTE OUTPUT, then upload matching_results.tsv to the Portal.
import pandas as pd
m = pd.read_csv("output/matching_results.tsv", sep="\t")
print(m.shape)
nm = m["matched_entity_ids"].fillna("").apply(lambda s: 0 if s == "" else s.count(",") + 1)
print("mean kept/S1:", round(float(nm.mean()), 3), "| empty %:", round(float((nm == 0).mean() * 100), 2))
print(nm.value_counts().sort_index().head(12).to_dict())
print(m.head(3).to_string())
!ls -lh output/matching_results.tsv
from google.colab import files
files.download("output/matching_results.tsv")


## What to send back

1. Scale-recall output (cell 4).
2. `threshold.json` + BEST line (cell 5).
3. Validator PASS + cell 7 stats.
4. **Public leaderboard score** after Portal upload.

Then we iterate: France analysis → error slices → targeted Exp6+.